In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    when,
    trim,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_real_estate_research_thesis(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "RealEstateResearchId string, SourceSystem string, PortfolioKey string, PortfolioName string,"
        #     "IngestionTimestamp timestamp, BatchId string, RealEstateResearchKey int,"
        #     "PartyKey int, RealEstateResearchStatusCode string, RealEstateResearchStatus string,"
        #     "RealEstateResearchDate timestamp, CreatedDate string, CompanyCode int, Company int,"
        #     "Comments string, OutcomeCode int, Outcome string, CancellationUser string,"
        #     "RecallDate timestamp,"
        #     "RealEstateResearchTypeCode string,"
        #     "RealEstateResearchType string,"
        #     "PacketCode int,"
        #     "Packet string,"
        #     "StreamCycleCode int,"
        #     "StreamCycle int,"
        #     "IsCompletedFlag boolean,"
        #     "LandRegistryCode string,"
        #     "LandRegistry string,"
        #     "LandOfficeCode string,"
        #     "LandOffice string,"
        #     "EAPSourceKey string,"
        #     "CancelledFlag int,"
        #     "CancelledDate timestamp,"
        #     "CancelledReasonCode string,"
        #     "CancelledReason string,"
        #     "TotalDays int,"
        #     "RealEstateResearchGroupStatusCode string,"
        #     "RealEstateResearchGroupStatus string,"
        #     "RealEstateResearchPerson int,"
        #     "RealEstateResearchPersonAddress string,"
        #     "RealEstateResearchPersonCounty string,"
        #     "RealEstateResearchRequestDate timestamp,"
        #     "RealEstateResearchAssignmentKey int,"
        #     "RealEstateResearchAssignmentOperator int,"
        #     "RealEstateResearchPriority boolean,"
        #     "RealEstateResearchRecallReason string,"
        #     "RealEstateResearchArea string,"
        #     "RealEstateResearchFirstDebtorCDI string,"
        #     "RealEstateResearchAssignmentPool string,"
        #     "RealEstateResearchRequestorComments string,"
        #     "RealEstateResearchPriorityReason string,"
        #     "RealEstateResearchAssignmentDate timestamp,"
        #     "RealEstateResearchRequestSystem string,"
        #     "RealEstateResearchReviewedDate timestamp,"
        #     "RealEstateResearchValidToDate timestamp,"
        #     "RealEstateResearcSubmittedBy int,"
        #     "RealEstateResearchPersonTaxRegistrationNumber string,"
        #     "HasRealEstateResearchProcessesed boolean,"
        #     "RealEstateResearchGroupCode int,"
        #     "RealEstateResearchRequestCode int,"
        #     "HasPerformedNewRealEstateResearchFlag int,"
        #     "RealEstateResearchExposureExpenses boolean,"
        #     "RealEstateResearchLastUpdatedDate timestamp,"
        #     "RealEstateKey int,"
        #     "RealEstateResearchExpiredDate timestamp,"
        #     "RealEstateResearchRiskStatus string,"
        #     "RealEstateResearchReferencekey string,"
        #     "RealEstateResearchRemediationStatus string,"
        #     "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.RealEstateResearch"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "thesis",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== thesis_preapproperty ==================

        thesis_preapproperty_extra_filters = None

        if thesis_preapproperty_extra_filters is not None:
            thesis_preapproperty_extra_filters.append(date_filter)
            thesis_preapproperty_final_filters = " AND ".join(
                thesis_preapproperty_extra_filters
            )
        else:
            thesis_preapproperty_final_filters = date_filter

        thesis_preapproperty_selected_fields = [
            "id",
            "rejection_reason",
            "risk_status",
            "reference_key",
            "remediation_status",
            "BusinessDate",
        ]

        thesis_preapproperty = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_preapproperty",
            thesis_preapproperty_final_filters,
            thesis_preapproperty_selected_fields,
        )

        # ================== thesis_preapdet ==================

        thesis_preapdet_extra_filters = None

        if thesis_preapdet_extra_filters is not None:
            thesis_preapdet_extra_filters.append(date_filter)
            thesis_preapdet_final_filters = " AND ".join(thesis_preapdet_extra_filters)
        else:
            thesis_preapdet_final_filters = date_filter

        thesis_preapdet_selected_fields = [
            "id",
            "status_code",
            "date_completed",
            "lawyer_code",
            "landregistry_id",
            "landoffice_code",
            "BusinessDate"
        ]

        thesis_preapdet = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_preapdet",
            thesis_preapdet_final_filters,
            thesis_preapdet_selected_fields,
        )

        # ================== thesis_preap ==================

        thesis_preap_extra_filters = None

        if thesis_preap_extra_filters is not None:
            thesis_preap_extra_filters.append(date_filter)
            thesis_preap_final_filters = " AND ".join(thesis_preap_extra_filters)
        else:
            thesis_preap_final_filters = date_filter

        thesis_preap_selected_fields = [
            "id",
            "portfolio_code",
            "crs",
            "status_code",
            "date_entered",
            "lawyer_id",
            "comments",
            "type_code",
            "reference_key",
            "contact_id",
            "address_id",
            "perif_enot_code",
            "request_date",
            "action_id",
            "operator_id",
            "priority",
            "recall_reason",
            "search_area",
            "first_debtor_crs",
            "assignment_pool",
            "group_status",
            "request_comments",
            "priority_reason",
            "date_requested",
            "request_system",
            "date_reviewed",
            "date_validto",
            "entered_by",
            "afm",
            "is_processed",
            "group_id",
            "request_code",
            "new_eap_inv",
            "loan_expense",
            "last_status_update",
            "due_date",
            "BusinessDate"
        ]

        thesis_preap = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_preap",
            thesis_preap_final_filters,
            thesis_preap_selected_fields,
        )
        

        # ================== JOINS ==================
        # Join conditions
        thesis_preapproperty_thesis_preapdet_join_condition = (
            thesis_preapproperty["id"]== thesis_preapdet["id"]
        )

        thesis_preapdet_thesis_preap_join_condition = (
            thesis_preapdet["id"] == thesis_preap["id"]
        )
        
        portfolio_thesis_preapproperty_join_condition = (
            thesis_preap['portfolio_code'] == portfolio['Prefix']
        )

        # Joins

        target_table_thesis = (
            thesis_preapproperty.join(
                thesis_preapdet,
                thesis_preapproperty_thesis_preapdet_join_condition,
                "inner",
            ).join(
                thesis_preap,
                thesis_preapdet_thesis_preap_join_condition,
                "inner",
            ).join(
                portfolio,
                portfolio_thesis_preapproperty_join_condition,
                "left",
            )
        )

        # HashKeys
        target_table_thesis_RealEstateResearchId_primary_key_fields = [
            thesis_preap["id"],
            thesis_preapproperty["id"],
            lit("thesis_RES"),
        ]

        target_table_thesis_RealEstateResearchKey_primary_key_fields = [
            thesis_preap["id"],
            thesis_preapproperty["id"],
        ]

        # Target table (hash keys and calculations)
        target_table_thesis = target_table_thesis.withColumn(
            "RealEstateResearchId",
            sha2(concat_ws(";", *target_table_thesis_RealEstateResearchId_primary_key_fields),256,),
        ).withColumn(
            "RealEstateResearchKey", concat_ws(";", *target_table_thesis_RealEstateResearchKey_primary_key_fields)
        ).withColumn(
            "IsCompletedFlag",
            when(thesis_preapdet.date_completed.isNotNull(), True).otherwise(False)
        )

        
        # ================== Selection of Final Columns ==================

        result_df_thesis = target_table_thesis.select(
            target_table_thesis.RealEstateResearchId.alias("RealEstateResearchId"),
            lit("thesis_RES").cast("string").alias("SourceSystem"),
            thesis_preap.portfolio_code.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).cast("timestamp").alias("IngestionTimestamp"),
            lit(None).cast("string").alias("BatchId"),
            target_table_thesis.RealEstateResearchKey.cast("int").alias("RealEstateResearchKey"),
            thesis_preap.crs.cast("int").alias("PartyKey"),
            thesis_preap.status_code.cast("string").alias("RealEstateResearchStatusCode"),
            thesis_preap.status_code.cast("string").alias("RealEstateResearchStatus"),
            thesis_preapdet.date_completed.cast("timestamp").alias(
                "RealEstateResearchDate"
            ),
            thesis_preap.date_entered.cast("string").alias("CreatedDate"),
            thesis_preapdet.lawyer_code.cast("int").alias("CompanyCode"),
            lit(None).cast("int").alias("Company"),
            thesis_preap.comments.cast("string").alias("Comments"),
            lit(None).cast("int").alias("OutcomeCode"),
            lit(None).cast("string").alias("Outcome"),
            lit(None).cast("string").alias(
                "CancellationUser"
            ),
            lit(None).alias("RecallDate"),
            thesis_preap.type_code.cast("string").alias(
                "RealEstateResearchTypeCode"
            ),
            thesis_preap.type_code.cast("string").alias(
                "RealEstateResearchType"
            ),
            lit(None).cast("int").alias("PacketCode"),
            lit(None).cast("string").alias("Packet"),
            lit(None).cast("int").alias("StreamCycleCode"),
            lit(None).cast("int").alias("StreamCycle"),
            target_table_thesis.IsCompletedFlag.cast("boolean").alias(
                "IsCompletedFlag"
            ),
            thesis_preapdet.landregistry_id.cast("string").alias(
                "LandRegistryCode"
            ),
            thesis_preapdet.landregistry_id.cast("string").alias(
                "LandRegistry"
            ),
            thesis_preapdet.landoffice_code.cast("string").alias(
                "LandOfficeCode"
            ),
            lit(None).cast("string").alias(
                "LandOffice"
            ),
            thesis_preap.reference_key.cast("string").alias("EAPSourceKey"),
            lit(None).cast("int").alias("CancelledFlag"),
            lit(None).cast("timestamp").alias("CancelledDate"),
            thesis_preapproperty.rejection_reason.cast("string").alias(
                "CancelledReasonCode"
            ),
            thesis_preapproperty.rejection_reason.cast("string").alias(
                "CancelledReason"
            ),
            lit(None).cast("int").alias("TotalDays"),
            thesis_preapdet.status_code.cast("string").alias("RealEstateResearchGroupStatusCode"),
            thesis_preap.group_status.cast("string").alias("RealEstateResearchGroupStatus"),
            thesis_preap.contact_id.cast("int").alias("RealEstateResearchPerson"),
            thesis_preap.address_id.cast("string").alias("RealEstateResearchPersonAddress"),
            thesis_preap.perif_enot_code.cast("string").alias("RealEstateResearchPersonCounty"),
            thesis_preap.request_date.cast("timestamp").alias("RealEstateResearchRequestDate"),
            thesis_preap.action_id.cast("int").alias("RealEstateResearchAssignmentKey"),
            thesis_preap.operator_id.cast("int").alias("RealEstateResearchAssignmentOperator"),
            thesis_preap.priority.cast("boolean").alias("RealEstateResearchPriority"),
            thesis_preap.recall_reason.cast("string").alias("RealEstateResearchRecallReason"),
            thesis_preap.search_area.cast("string").alias("RealEstateResearchArea"),
            thesis_preap.first_debtor_crs.cast("string").alias("RealEstateResearchFirstDebtorCDI"),
            thesis_preap.assignment_pool.cast("string").alias("RealEstateResearchAssignmentPool"),
            thesis_preap.request_comments.cast("string").alias("RealEstateResearchRequestorComments"),
            thesis_preap.priority_reason.cast("string").alias("RealEstateResearchPriorityReason"),
            thesis_preap.date_requested.cast("timestamp").alias("RealEstateResearchAssignmentDate"),
            thesis_preap.request_system.cast("string").alias("RealEstateResearchRequestSystem"),
            thesis_preap.date_reviewed.cast("timestamp").alias("RealEstateResearchReviewedDate"),
            thesis_preap.date_validto.cast("timestamp").alias("RealEstateResearchValidToDate"),
            thesis_preap.entered_by.cast("int").alias("RealEstateResearcSubmittedBy"),
            thesis_preap.afm.cast("string").alias("RealEstateResearchPersonTaxRegistrationNumber"),
            thesis_preap.is_processed.cast("boolean").alias("HasRealEstateResearchProcessesed"),
            thesis_preap.group_id.cast("int").alias("RealEstateResearchGroupCode"),
            thesis_preap.request_code.cast("int").alias("RealEstateResearchRequestCode"),
            thesis_preap.new_eap_inv.cast("int").alias("HasPerformedNewRealEstateResearchFlag"),
            thesis_preap.loan_expense.cast("boolean").alias("RealEstateResearchExposureExpenses"),
            thesis_preap.last_status_update.cast("timestamp").alias("RealEstateResearchLastUpdatedDate"),
            thesis_preapproperty.id.cast("int").alias("RealEstateKey"),
            thesis_preap.due_date.cast("timestamp").alias("RealEstateResearchExpiredDate"),
            thesis_preapproperty.risk_status.cast("string").alias("RealEstateResearchRiskStatus"),
            thesis_preapproperty.reference_key.cast("string").alias("RealEstateResearchReferencekey"),
            thesis_preapproperty.remediation_status.cast("string").alias("RealEstateResearchRemediationStatus"),
            thesis_preap.BusinessDate.cast("date").alias("BusinessDate"),
        )


        # ================== Append ==================
        
        result_df_thesis.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_real_estate_research_thesis(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)